In [1]:
!pip install librosa soundfile numpy pydub tqdm



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import librosa
import soundfile as sf
import numpy as np
from tqdm import tqdm
import random

# Paths
clean_path = r"C:\Users\Hp\OneDrive\Documents\notch_project\data\wav_clean"
noise_root = r"C:\Users\Hp\OneDrive\Documents\notch_project\data\wav_noise"
output_path = r"C:\Users\Hp\OneDrive\Documents\notch_project\data\mixed"

os.makedirs(output_path, exist_ok=True)

# Load noise files
noise_files = []
for subfolder in ["glass", "babble"]:
    folder = os.path.join(noise_root, subfolder)
    for file in os.listdir(folder):
        if file.endswith(".wav"):
            noise_files.append(os.path.join(folder, file))

print(f"Loaded {len(noise_files)} noise files.")

# Mix function
def mix_audio(clean_file, noise_file, snr_db=5):
    clean, sr = librosa.load(clean_file, sr=None)
    noise, _ = librosa.load(noise_file, sr=sr)
    
    # Repeat or trim noise to match length of speech
    if len(noise) < len(clean):
        noise = np.tile(noise, int(np.ceil(len(clean)/len(noise))))
    noise = noise[:len(clean)]
    
    # Adjust noise volume based on SNR
    clean_rms = np.sqrt(np.mean(clean**2))
    noise_rms = np.sqrt(np.mean(noise**2))
    desired_noise_rms = clean_rms / (10**(snr_db/20))
    noise = noise * (desired_noise_rms / (noise_rms + 1e-6))
    
    mixed = clean + noise
    return mixed, sr

# Mix clean and random noise
for file in tqdm(os.listdir(clean_path)):
    if file.endswith(".wav"):
        clean_file = os.path.join(clean_path, file)
        noise_file = random.choice(noise_files)
        mixed, sr = mix_audio(clean_file, noise_file, snr_db=random.choice([0, 5, 10]))
        out_file = os.path.join(output_path, "noisy_" + file)
        sf.write(out_file, mixed, sr)

print(f"✅ Mixed audio saved to: {output_path}")


Loaded 6 noise files.


100%|████████████████████████| 2703/2703 [02:41<00:00, 16.75it/s]

✅ Mixed audio saved to: C:\Users\Hp\OneDrive\Documents\notch_project\data\mixed


In [3]:
# run in a terminal or a notebook cell with a "!" prefix
!pip install librosa soundfile scikit-learn joblib matplotlib pandas tqdm



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import glob
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib
import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
!pip install seaborn



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# change these if your folders are in different locations
CLEAN_DIR = r"C:\Users\Hp\OneDrive\Documents\notch_project\data\wav_clean"
MIXED_DIR = r"C:\Users\Hp\OneDrive\Documents\notch_project\data\mixed"   # or wherever you stored noisy files
# If you arranged noise types by subfolders inside mixed, we will detect them automatically.

print("CLEAN_DIR:", CLEAN_DIR)
print("MIXED_DIR:", MIXED_DIR)


CLEAN_DIR: C:\Users\Hp\OneDrive\Documents\notch_project\data\wav_clean
MIXED_DIR: C:\Users\Hp\OneDrive\Documents\notch_project\data\mixed


In [7]:
def collect_files(clean_dir, mixed_dir):
    files = []
    # clean
    for f in glob.glob(os.path.join(clean_dir, "*.wav")):
        files.append((f, "clean"))
    # mixed: expect subfolders for noise type
    for noise_folder in sorted(os.listdir(mixed_dir)):
        folder_path = os.path.join(mixed_dir, noise_folder)
        if os.path.isdir(folder_path):
            for f in glob.glob(os.path.join(folder_path, "*.wav")):
                files.append((f, noise_folder))   # label = subfolder name (glass, babble)
    return files

files = collect_files(CLEAN_DIR, MIXED_DIR)
print(f"Collected {len(files)} files. Example:", files[:3])


Collected 2703 files. Example: [('C:\\Users\\Hp\\OneDrive\\Documents\\notch_project\\data\\wav_clean\\1272-128104-0000.wav', 'clean'), ('C:\\Users\\Hp\\OneDrive\\Documents\\notch_project\\data\\wav_clean\\1272-128104-0001.wav', 'clean'), ('C:\\Users\\Hp\\OneDrive\\Documents\\notch_project\\data\\wav_clean\\1272-128104-0002.wav', 'clean')]


In [10]:
def safe_extract(path):
    try:
        fv = extract_features(path, n_mfcc=n_mfcc, sr=sr_target, max_duration=max_duration)
        return fv
    except Exception as e:
        print("ERR:", path, "->", e)
        return None


In [11]:
import glob

clean_files = glob.glob("data/wav_clean/**/*.wav", recursive=True)
noise_files = glob.glob("data/wav_noise/**/*.wav", recursive=True)

print("✅ Found", len(clean_files), "clean files and", len(noise_files), "noise files.")


✅ Found 2703 clean files and 6 noise files.


In [23]:
# parameters
n_mfcc = 13
sr_target = 16000       # LibriSpeech = 16 kHz
max_duration = None     # or e.g., 6.0 to limit length

def extract_features(filename, n_mfcc=13, sr=sr_target, max_duration=None):
    # load (librosa will resample to sr)
    y, sr_loaded = librosa.load(filename, sr=sr)  # returns audio and sampling rate
    if max_duration is not None:
        max_samples = int(max_duration * sr)
        if len(y) > max_samples:
            y = y[:max_samples]
    # MFCCs -> (n_mfcc, T)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std  = mfcc.std(axis=1)
    feat = np.concatenate([mfcc_mean, mfcc_std])  # length = 2 * n_mfcc
    return feat

# quick sanity test (replace with an actual path if files is not defined)
# print("Example feature shape:", extract_features(clean_files[0], n_mfcc=n_mfcc).shape)

In [24]:
from tqdm import tqdm
import numpy as np

X = []
y = []

print("clean files:", len(clean_files), "noise files:", len(noise_files))

for p in tqdm(clean_files, desc="clean"):
    fv = safe_extract(p)
    if fv is not None:
        X.append(fv)
        y.append("clean")

for p in tqdm(noise_files, desc="noisy"):
    fv = safe_extract(p)
    if fv is not None:
        X.append(fv)
        y.append("noisy")

X = np.stack(X)
y = np.array(y)
print("🎯 Done! X shape:", X.shape, "y shape:", y.shape)


clean files: 2703 noise files: 6


noisy: 100%|███████████████████████| 6/6 [00:00<00:00, 23.69it/s]

🎯 Done! X shape: (2709, 26) y shape: (2709,)


In [14]:
X = []
y = []
failed = []
for filepath, label in tqdm(files):
    try:
        feat = extract_features(filepath, n_mfcc=n_mfcc, sr=sr_target)
        X.append(feat)
        y.append(label)
    except Exception as e:
        failed.append((filepath, str(e)))

X = np.array(X)
y = np.array(y)
print("X shape:", X.shape, "labels:", np.unique(y))
if failed:
    print("Failed files:", failed[:5])


100%|████████████████████████| 2703/2703 [00:58<00:00, 45.90it/s]

X shape: (2703, 26) labels: ['clean']


In [15]:
# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# pipeline
clf = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=42))

print("Training...")
clf.fit(X_train, y_train)

# evaluate
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred))


Training...


ValueError: The number of classes has to be greater than one; got 1 class

In [16]:
# Safe audio loader + feature extractor for librosa
import numpy as np
import soundfile as sf
import librosa

def load_audio_safe(path, sr=16000, mono=True):
    # read with soundfile -> returns float64 by default (floating point OK)
    y, orig_sr = sf.read(path, dtype='float32')   # float32 -> ok for librosa
    # If stereo, convert to mono by averaging channels
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    # Resample if needed
    if orig_sr != sr:
        y = librosa.resample(y, orig_sr, sr)
    # ensure 1D float32
    y = np.asarray(y, dtype=np.float32)
    return y, sr

def ensure_min_length(y, sr, n_fft=2048):
    # pad with zeros if too short for n_fft
    min_len = n_fft
    if len(y) < min_len:
        pad = min_len - len(y)
        y = np.pad(y, (0, pad), mode='constant', constant_values=0.0)
    return y

def extract_features(path, sr=16000, n_mfcc=13, n_fft=2048, hop_length=512):
    y, sr = load_audio_safe(path, sr=sr, mono=True)
    y = ensure_min_length(y, sr, n_fft=n_fft)

    # MFCCs
    mf = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
    d1 = librosa.feature.delta(mf)
    d2 = librosa.feature.delta(mf, order=2)

    # spectral contrast and centroid
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length)

    # aggregate: mean and std for each feature row
    feats = np.concatenate([
        mf.mean(axis=1), mf.std(axis=1),
        d1.mean(axis=1), d1.std(axis=1),
        d2.mean(axis=1), d2.std(axis=1),
        contrast.mean(axis=1), contrast.std(axis=1),
        centroid.mean(axis=1), centroid.std(axis=1),
    ])
    return feats


In [17]:
# example
f = extract_features("data/wav_clean/1272-128104-0000.wav", sr=8000)  # or sr=16000
print(f.shape)  # feature vector shape


TypeError: resample() takes 1 positional argument but 3 were given

In [18]:
import numpy as np
import soundfile as sf
import librosa

def load_audio_safe(path, sr=16000, mono=True):
    # read as float32 so librosa features are happy
    y, orig_sr = sf.read(path, dtype='float32')
    # stereo -> mono
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    # resample: use keyword names (some librosa versions require this)
    if orig_sr != sr:
        y = librosa.resample(y=y, orig_sr=orig_sr, target_sr=sr)
    y = np.asarray(y, dtype=np.float32)
    return y, sr


In [19]:
y, orig_sr = sf.read(path, dtype='float32')
print("dtype, shape, sr:", y.dtype, y.shape, orig_sr)


NameError: name 'path' is not defined

In [20]:
import soundfile as sf

path = "data/wav_clean/1272-128104-0000.wav"  # your test wav file
y, orig_sr = sf.read(path, dtype='float32')
print("dtype, shape, sr:", y.dtype, y.shape, orig_sr)


dtype, shape, sr: float32 (93680,) 16000


In [21]:
f = extract_features("data/wav_clean/1272-128104-0000.wav", sr=16000)
print(f.shape)


(94,)


In [ ]:
(94,)


In [22]:
import os, glob
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd
import joblib

# adjust these if needed
CLEAN_GLOB = "data/wav_clean/**/*.wav"
NOISE_GLOB = "data/wav_noise/**/*.wav"
SR = 16000  # target sample rate used earlier

# get file lists
clean_files = glob.glob(CLEAN_GLOB, recursive=True)
noise_files = glob.glob(NOISE_GLOB, recursive=True)
print("clean files:", len(clean_files), "noise files:", len(noise_files))

# function you already wrote: extract_features(path, sr=16000, ...)
# assume it returns 1D feature vector (shape (94,) in your test)
# We'll wrap it and handle exceptions gracefully.
def safe_extract(path):
    try:
        y, sr = load_audio_safe(path, sr=8000)
        fv = extract_features(path, sr=sr)  # <-- use your real function name here
        return fv
    except Exception as e:
        print("ERR:", path, "->", e)
        return None


# Build dataset
X = []
y = []
# clean
for p in tqdm(clean_files, desc="clean"):
    fv = safe_extract(p)
    if fv is not None:
        X.append(fv)
        y.append("clean")

# noisy: we assume you mixed noises and created noisy files OR you plan to mix on the fly.
# If you already have mixed noisy files folder, use that. Otherwise create noisy files first.
for p in tqdm(noise_files, desc="noisy"):
    fv = safe_extract(p)
    if fv is not None:
        X.append(fv)
        y.append("noisy")

X = np.stack(X)
y = np.array(y)
print("X shape:", X.shape, "y shape:", y.shape)


clean files: 2703 noise files: 6


clean:   0%|          | 0/2703 [00:00<?, ?it/s]

ERR: data/wav_clean\1272-128104-0000.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_clean\1272-128104-0001.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_clean\1272-128104-0002.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_clean\1272-128104-0003.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_clean\1272-128104-0004.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_clean\1272-128104-0005.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_clean\1272-128104-0006.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_clean\1272-128104-0007.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_clean\1272-128104-0008.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_clean\1272-128104-0009.

noisy:   0%|          | 0/6 [00:00<?, ?it/s]

ERR: data/wav_noise\babble\27762_lg_noise09.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_noise\babble\crowd_noise.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_noise\babble\people_talking.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_noise\glass\521341_zxaudiocraft_zxaudiocraft-glass-break-005.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_noise\glass\bottle_shatter.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.
ERR: data/wav_noise\glass\glass_break.wav -> Frequency band exceeds Nyquist. Reduce either fmin or n_bands.


ValueError: need at least one array to stack

In [ ]:
!pip install ipywidgets tqdm --upgrade


In [ ]:
!jupyter nbextension enable --py widgetsnbextension --sys-prefix


In [ ]:
pip install ipywidgets
from tqdm.notebook import tqdm


In [ ]:
!pip install ipywidgets tqdm --upgrade


In [ ]:
!jupyter nbextension enable --py widgetsnbextension --sys-prefix


In [ ]:
from tqdm.notebook import tqdm


In [ ]:
from tqdm import tqdm


In [ ]:
from tqdm import tqdm


In [ ]:
from tqdm import tqdm
for i in tqdm(range(5)):
    pass


In [ ]:
import soundfile as sf
import librosa
import numpy as np

def load_audio_safe(path, sr=16000, mono=True):
    y, orig_sr = sf.read(path, dtype='float32')
    if mono and y.ndim > 1:
        y = np.mean(y, axis=1)
    if orig_sr != sr:
        # Fix for new librosa versions (use keyword args)
        y = librosa.resample(y=y, orig_sr=orig_sr, target_sr=sr)
    y = np.asarray(y, dtype=np.float32)
    return y, sr


In [ ]:
test_path = "data/wav_clean/1272-128104-0000.wav"
y, sr = load_audio_safe(test_path, sr=8000)
print("dtype:", y.dtype)
print("shape:", y.shape)
print("sr:", sr)


In [ ]:
from tqdm import tqdm
X = []
y = []

print("clean files:", len(clean_files), "noise files:", len(noise_files))

for p in tqdm(clean_files, desc="clean"):
    fv = safe_extract(p)
    if fv is not None:
        X.append(fv)
        y.append("clean")

for p in tqdm(noise_files, desc="noisy"):
    fv = safe_extract(p)
    if fv is not None:
        X.append(fv)
        y.append("noisy")

X = np.stack(X)
y = np.array(y)
print("Done. X shape:", X.shape, "y shape:", y.shape)


In [ ]:
import glob
clean_files = glob.glob("data/wav_clean/**/*.wav", recursive=True)
noise_files = glob.glob("data/wav_noise/**/*.wav", recursive=True)


In [ ]:
import librosa
import numpy as np

# parameters (you can change these)
n_mfcc = 13
sr_target = 16000
max_duration = None   # or set to e.g. 6.0 seconds to truncate

def extract_features(filename, n_mfcc=13, sr=sr_target, max_duration=None):
    """
    Load audio, resample to sr, optionally truncate, compute MFCC mean+std -> 2*n_mfcc vector
    """
    y, sr_loaded = librosa.load(filename, sr=sr)   # librosa will resample to sr
    if max_duration is not None:
        max_samples = int(max_duration * sr)
        if len(y) > max_samples:
            y = y[:max_samples]
    # compute MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std  = mfcc.std(axis=1)
    feat = np.concatenate([mfcc_mean, mfcc_std])   # length = 2 * n_mfcc
    return feat


In [ ]:
def safe_extract(path):
    try:
        return extract_features(path, n_mfcc=n_mfcc, sr=sr_target, max_duration=max_duration)
    except Exception as e:
        print("ERR:", path, "->", e)
        return None


In [ ]:
from tqdm import tqdm
import numpy as np

X = []
y = []

print("clean files:", len(clean_files), "noise files:", len(noise_files))

for p in tqdm(clean_files, desc="clean"):
    fv = safe_extract(p)
    if fv is not None:
        X.append(fv)
        y.append("clean")

for p in tqdm(noise_files, desc="noisy"):
    fv = safe_extract(p)
    if fv is not None:
        X.append(fv)
        y.append("noisy")

X = np.stack(X)
y = np.array(y)
print("Done. X shape:", X.shape, "y shape:", y.shape)


In [ ]:
# 1) split, pipeline (scaler+SVM), train, evaluate, save
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1.a split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("train/test sizes:", X_train.shape, X_test.shape)

# 1.b pipeline
clf = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=42))

# 1.c train
print("Training SVM...")
clf.fit(X_train, y_train)

# 1.d predict + metrics
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred))

# 1.e confusion matrix plot
cm = confusion_matrix(y_test, y_pred, labels=np.unique(y))
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=np.unique(y), yticklabels=np.unique(y))
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix")
plt.show()

# 1.f save model
joblib.dump(clf, "notch_svm_pipeline.joblib")
print("Saved model -> notch_svm_pipeline.joblib")


In [25]:
# ONE CELL: rebuild features + train pipeline
# Paste this into one cell and RUN (may take a few minutes)

# --- imports ---
import os, glob, warnings
warnings.filterwarnings("ignore")
import numpy as np
import soundfile as sf
import librosa
from tqdm import tqdm

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# --- parameters ---
sr_target = 16000   # sampling rate used for feature extraction (match your previous runs)
n_mfcc = 13
max_duration = None   # or set like 6.0 to limit length per sample

# --- helper: safe audio load that handles stereo/mono + resample if needed ---
def load_audio_safe(path, sr=sr_target, mono=True):
    # read with soundfile (keeps dtype)
    try:
        y, orig_sr = sf.read(path, dtype='float32')
    except Exception as e:
        print("ERR reading:", path, "->", e)
        return None, None
    # if stereo convert to mono
    if y is None:
        return None, None
    if y.ndim > 1:
        y = np.mean(y, axis=1)
    # resample with librosa if needed
    if orig_sr != sr:
        y = librosa.resample(y, orig_sr, sr)
    y = np.asarray(y, dtype=np.float32)
    return y, sr

# --- feature extractor (same summary MFCC features as before) ---
def extract_features_from_audio(y, sr=sr_target, n_mfcc=n_mfcc, max_duration=None):
    if y is None:
        return None
    if max_duration is not None:
        max_samples = int(max_duration * sr)
        y = y[:max_samples]
    # compute MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std  = mfcc.std(axis=1)
    feat = np.concatenate([mfcc_mean, mfcc_std])
    return feat

# --- safe_extract wrapper: given a filepath returns feature vector or None ---
def safe_extract(path):
    y, sr = load_audio_safe(path, sr=sr_target)
    if y is None:
        return None
    feat = extract_features_from_audio(y, sr=sr_target, n_mfcc=n_mfcc, max_duration=max_duration)
    return feat

# --- find files (adjust the patterns if your folders differ) ---
clean_files = glob.glob("data/wav_clean/**/*.wav", recursive=True)
noise_files = glob.glob("data/wav_noise/**/*.wav", recursive=True)

print("clean files:", len(clean_files), "noise files:", len(noise_files))
if len(clean_files)==0:
    raise RuntimeError("No clean files found in data/wav_clean. Check path.")

# --- build dataset X,y ---
X = []
y = []

print("Extracting features (clean)...")
for p in tqdm(clean_files, desc="clean"):
    fv = safe_extract(p)
    if fv is not None:
        X.append(fv)
        y.append("clean")

print("Extracting features (noisy)...")
# If you already created noisy mixed files, point to that folder instead.
# Here we assume you have a small folder of pre-mixed noisy examples or the short noise files
for p in tqdm(noise_files, desc="noisy"):
    fv = safe_extract(p)
    if fv is not None:
        X.append(fv)
        y.append("noisy")

X = np.stack(X)
y = np.array(y)
print("🎯 Done! X shape:", X.shape, "y shape:", y.shape)

# Save features for quick reload next time
np.savez("dataset_features.npz", X=X, y=y)
print("Saved dataset_features.npz")

# --- Train/test split and SVM pipeline ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("train/test sizes:", X_train.shape, X_test.shape)

clf = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=42))

print("Training SVM...")
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=np.unique(y))
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=np.unique(y), yticklabels=np.unique(y))
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix")
plt.show()

# save pipeline
joblib.dump(clf, "notch_svm_pipeline.joblib")
print("Saved model -> notch_svm_pipeline.joblib")


clean files: 2703 noise files: 6
Extracting features (clean)...


clean: 100%|█████████████████| 2703/2703 [00:54<00:00, 49.99it/s]


Extracting features (noisy)...


noisy:   0%|                               | 0/6 [00:00<?, ?it/s]


TypeError: resample() takes 1 positional argument but 3 were given